# Ekstrakcja i Agregacja Danych z Platformy Steam
**Autor:** Konrad Pajor
**Projekt:** Praca licencjacka – Analiza sentymentu recenzji gier wideo

Poniższy notatnik służy do zautomatyzowanego pobierania danych empirycznych przy użyciu oficjalnego API platformy Steam (biblioteka `steamreviews`).

Celem skryptu jest:
1. Pobranie surowych opinii dla wybranych, zróżnicowanych gatunkowo gier.
2. Wstępna kategoryzacja recenzji pod kątem trollingu i sarkazmu (wyliczenie wskaźnika proporcji głosów społecznościowych `Ratio`).
3. Spłaszczenie struktury JSON i eksport danych do surowych plików `.xlsx`, które posłużą jako strumień wejściowy dla głównego modelu NLP.

## 1. Import bibliotek

In [ ]:
import steamreviews
import pandas as pd
from tqdm.notebook import tqdm
import os

## 2. Definicja głównej funkcji pobierającej

W poniższej komórce zdefiniowano uniwersalną funkcję `pobierz_recenzje_steam()`. Zgodnie z dobrymi praktykami inżynierii oprogramowania (zasada DRY - *Don't Repeat Yourself*), logika pobierania, obliczania wskaźników i zapisu została zintegrowana w jednym miejscu. Zabezpiecza to proces przed nadpisywaniem plików i ułatwia ewentualne skalowanie projektu o kolejne tytuły gier.

In [ ]:
def pobierz_recenzje_steam(app_id, nazwa_gry, plik_wyjsciowy, jezyki=['polish']):
    print(f"\n{'='*60}")
    print(f"Rozpoczynamy pobieranie recenzji dla: {nazwa_gry} (ID: {app_id})")
    print(f"{'='*60}")

    wszystkie_recenzje = {}

    # Pobieramy dane oddzielnie dla kazdego jezyka i laczymy je
    for lang in jezyki:
        print(f"-> Pobieranie paczki dla jezyka: {lang.upper()}...")
        request_params = dict(language=lang)

        # verbose=True gwarantuje komunikaty z samej biblioteki steamreviews
        review_dict, _ = steamreviews.download_reviews_for_app_id(
            app_id,
            chosen_request_params=request_params,
            verbose=True
        )

        if 'reviews' in review_dict:
            wszystkie_recenzje.update(review_dict['reviews'])

    print(f"\nZebrano lacznie {len(wszystkie_recenzje)} recenzji. Przetwarzanie...")

    data_for_excel = []

    # Dynamiczny pasek postepu dostosowany do Jupyter Notebook
    for review_id, review_data in tqdm(wszystkie_recenzje.items(), desc=f"Analiza {nazwa_gry}"):
        v_funny = review_data.get('votes_funny', 0)
        v_up = review_data.get('votes_up', 0)
        text = review_data.get('review', '')
        is_positive = review_data.get('voted_up', True)
        jezyk_recenzji = review_data.get('language', 'unknown')

        # Logika wspolczynnika
        if v_up > 0:
            ratio = v_funny / v_up
        else:
            ratio = 1.0 if v_funny > 0 else 0.0

        # Klasyfikacja (wykrywanie trolli)
        if ratio > 0.7:
            kategoria = "TROLL / CZYSTY SARKAZM"
        elif ratio > 0.5:
            kategoria = "ZABAWNA ALE PRZYDATNA"
        else:
            kategoria = "NORMALNA"

        row = {
            'ID Recenzji': review_id,
            'Jezyk': jezyk_recenzji,
            'Kategoria': kategoria,
            'Wspolczynnik Funny/Up': round(ratio, 2),
            'Glosy Funny': v_funny,
            'Glosy Helpful': v_up,
            'Ocena': 'Pozytywna' if is_positive else 'Negatywna',
            'Tresc Recenzji': text
        }
        data_for_excel.append(row)

    # Tworzymy DataFrame
    df = pd.DataFrame(data_for_excel)

    # Zabezpieczenie: tworzymy folder docelowy, jesli nie istnieje
    katalog = os.path.dirname(plik_wyjsciowy)
    if katalog:
        os.makedirs(katalog, exist_ok=True)

    # Zapis
    df.to_excel(plik_wyjsciowy, index=False)
    print(f"Sukces! Zapisano plik: {plik_wyjsciowy}\n")

## 3. Konfiguracja zbioru badawczego i pętla wykonawcza

Zdefiniowano słownik zawierający identyfikatory API (`app_id`) wybranych gier (m.in. wysokobudżetowe produkcje AAA takie jak *Cyberpunk 2077* czy *Red Dead Redemption 2*, a także produkcje niezależne).

Pętla iteruje po wszystkich tytułach, pobierając recenzje w języku polskim (z wyjątkiem gry Beholder). Wynikowe pliki arkusza kalkulacyjnego są dynamicznie zapisywane w dedykowanym folderze `/data/`.

In [ ]:
# Słownik gier w formacie: {ID: ("Nazwa", "Docelowy plik_excel")}
gry_do_pobrania = {
    323190:  ("Frostpunk", "../data/recenzje_steam_analiza.xlsx"),
    282070:  ("This War of Mine", "../data/recenzje_steam_analiza2.xlsx"),
    292030:  ("Wiedźmin 3: Dziki Gon", "../data/recenzje_steam_analiza3.xlsx"),
    1086940: ("Baldur's Gate 3", "../data/recenzje_steam_analiza4.xlsx"), 
    475550:  ("Beholder", "../data/recenzje_steam_analiza5.xlsx"),
    1091500: ("Cyberpunk 2077", "../data/recenzje_steam_analiza6.xlsx"),
    1174180: ("Red Dead Redemption 2", "../data/recenzje_steam_analiza7.xlsx")
}

# Główna pętla wykonawcza
for app_id, (nazwa, plik) in gry_do_pobrania.items():
    if app_id==475550:
        lista_jezykow = ['polish','english']
        pobierz_recenzje_steam(app_id, nazwa, plik, jezyki=lista_jezykow)
    else:
        lista_jezykow = ['polish']
        pobierz_recenzje_steam(app_id, nazwa, plik, jezyki=lista_jezykow)